In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from collections import Counter

from src.twitter_data_prep import (
    clean_twitter_data, add_engagement_features,
    add_time_features, add_pre_post_chatgpt, aggregate_by_month
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

OUT_FIG  = os.path.abspath('../../reports/twitter_reports')
OUT_DATA = os.path.abspath('../../data/processed/twitter_processed')
os.makedirs(OUT_FIG,  exist_ok=True)
os.makedirs(OUT_DATA, exist_ok=True)
print('Setup OK')

Setup OK


## 1. Load & Clean

In [2]:
raw = pd.read_csv('../../data/raw/twitter_raw/twitter_raw.csv', parse_dates=['date'])
print(f'Raw: {raw.shape}')

df = clean_twitter_data(raw)
df = add_engagement_features(df)
df = add_time_features(df)
df = add_pre_post_chatgpt(df)

print(f'Clean: {df.shape}')
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/twitter_raw/twitter_raw.csv'

## 2. Monthly Tweet Volume

In [ ]:
monthly = aggregate_by_month(df)

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#2196F3' if r else '#BBDEFB' for r in monthly['reliable']]
ax.bar(monthly['month'], monthly['tweet_count'], color=colors, width=20, edgecolor='none')
ax.axvline(pd.Timestamp('2022-11-30'), color='crimson', ls='--', lw=1.5, label='ChatGPT launch')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.set_title('Monthly Tweet Volume — Wikipedia Topics', fontsize=14, weight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Tweet Count')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_monthly_volume.png', dpi=150)
plt.show()
print(f'Peak month: {monthly.loc[monthly.tweet_count.idxmax(), "month"].strftime("%b %Y")} — {monthly.tweet_count.max()} tweets')

## 3. Engagement Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = [('likes', '#4CAF50'), ('retweets', '#FF9800'), ('replies', '#9C27B0')]

for ax, (col, color) in zip(axes, metrics):
    data = df[col][df[col] > 0]
    ax.hist(np.log1p(data), bins=40, color=color, alpha=0.85, edgecolor='none')
    ax.set_title(f'{col.capitalize()} (log scale)', fontsize=12)
    ax.set_xlabel('log(1 + count)')
    ax.set_ylabel('# Tweets')
    ax.axvline(np.log1p(data.median()), color='black', lw=1.5, ls='--', label=f'median={data.median():.0f}')
    ax.legend(fontsize=9)

plt.suptitle('Engagement Distributions (log-scale)', fontsize=13, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_engagement_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Engagement Tier Breakdown

In [ ]:
tier_counts = df['engagement_tier'].value_counts().reindex(['viral','high','medium','low'])

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#D32F2F','#F57C00','#388E3C','#1976D2']
bars = ax.barh(tier_counts.index, tier_counts.values, color=colors, edgecolor='none')
for bar, val in zip(bars, tier_counts.values):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_title('Tweet Engagement Tier Distribution', fontsize=13, weight='bold')
ax.set_xlabel('Number of Tweets')
ax.set_xlim(0, tier_counts.max() * 1.15)
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_engagement_tiers.png', dpi=150)
plt.show()

## 5. Top Hashtags

In [ ]:
all_tags = [tag for tags in df['hashtags'] for tag in (tags if isinstance(tags, list) else [])]
tag_counts = Counter(all_tags)

# Filter out 'wikipedia' itself as it's trivially dominant
top_tags = pd.DataFrame(
    [(t, c) for t, c in tag_counts.most_common(25) if t != 'wikipedia'],
    columns=['hashtag','count']
)

fig, ax = plt.subplots(figsize=(8, 6))
palette = sns.color_palette('Blues_r', n_colors=len(top_tags))
ax.barh(top_tags['hashtag'][::-1], top_tags['count'][::-1], color=palette, edgecolor='none')
ax.set_title('Top 25 Hashtags (excl. #wikipedia)', fontsize=13, weight='bold')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_top_hashtags.png', dpi=150)
plt.show()

print(top_tags.head(10).to_string(index=False))

## 6. Pre vs Post ChatGPT

In [ ]:
period_stats = df.groupby('period').agg(
    tweet_count    = ('id',              'count'),
    avg_likes      = ('likes',           'mean'),
    avg_retweets   = ('retweets',        'mean'),
    avg_engagement = ('total_engagement','mean'),
).round(2)

print(period_stats)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
metrics_pp = ['avg_likes', 'avg_retweets', 'avg_engagement']
titles_pp  = ['Avg Likes', 'Avg Retweets', 'Avg Total Engagement']
colors_pp  = ['#42A5F5', '#EF5350']

for ax, metric, title in zip(axes, metrics_pp, titles_pp):
    vals = period_stats.loc[['pre-ChatGPT','post-ChatGPT'], metric]
    ax.bar(vals.index, vals.values, color=colors_pp, edgecolor='none', width=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Average')
    for i, v in enumerate(vals.values):
        ax.text(i, v + vals.max()*0.02, f'{v:.1f}', ha='center', fontsize=10)

plt.suptitle('Pre vs Post ChatGPT — Engagement Comparison', fontsize=13, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_pre_post_chatgpt.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Most Viral Tweets

In [ ]:
viral = (
    df[df['engagement_tier'] == 'viral']
    .sort_values('total_engagement', ascending=False)
    [['date','text','likes','retweets','replies','total_engagement','url']]
    .head(10)
    .reset_index(drop=True)
)
viral['text_short'] = viral['text'].str[:120] + '...'
viral[['date','text_short','likes','retweets','total_engagement']]

## 8. Activity Heatmap — Day of Week × Hour

In [ ]:
days_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = (
    df.groupby(['day_of_week','hour'])
    .size()
    .unstack(fill_value=0)
    .reindex(days_order)
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    heatmap_data, ax=ax, cmap='YlOrRd',
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'Tweet Count'}
)
ax.set_title('Tweet Activity Heatmap — Day of Week vs Hour (UTC)', fontsize=13, weight='bold')
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_activity_heatmap.png', dpi=150)
plt.show()

## 9. Save Processed Data

In [ ]:
# Convert list columns to strings for CSV compatibility
df['hashtags'] = df['hashtags'].apply(lambda x: '|'.join(x) if isinstance(x, list) else '')
df['mentions'] = df['mentions'].apply(lambda x: '|'.join(x) if isinstance(x, list) else '')

out = os.path.join(OUT_DATA, 'twitter_clean.csv')
df.to_csv(out, index=False)
print(f'Saved → {out}')
print(f'Shape: {df.shape}')
df.dtypes